### Reading Files

In [1]:
using JuMP, Gurobi, Random, Distributions,DataFrames, LinearAlgebra
function read_orlib_cap(filepath::String)
    open(filepath) do io
        # 1) read n (#facilities) and m (#cities)
        n, m = parse.(Int, split(readline(io)))

        # 2) read facility data: capacity and opening cost
        capacity  = Vector{Float64}(undef, n)
        fixedcost = Vector{Float64}(undef, n)
        for i in 1:n
            # each line: "<capacity> <opening cost>"
            c1, c2 = parse.(Float64, split(readline(io)))
            capacity[i], fixedcost[i] = c1, c2
        end

        # 3) read city‐by‐city: demand + cost to each facility
        demand = Vector{Float64}(undef, m)
        cost = Array{Float64}(undef, n, m)
        for j in 1:m
            # read demand line
            demand[j] = parse(Float64, strip(readline(io)))
            # now read cost entries until we have n of them
            vals = Float64[]
            while length(vals) < n
                append!(vals, parse.(Float64, split(readline(io))))
            end
            cost[:,j] = vals[1:n]
        end

        return capacity, fixedcost, cost, demand
    end
end

#Reading files
file_path = "/Users/Patron/Documents/ORLIB/ORLIB-cap/90/cap91.txt" ## Add the path file here
capacity, fixedcost, cost, demandmean = read_orlib_cap(file_path);


### (Optional) code if one wants to perturb the facility location and capacity data

In [ ]:
#Random.seed!(1234)      
#nloc = length(fixedcost)

# choose your perturbation magnitude 
#delta = 0.5
#dist_eps = Uniform(-delta, delta)
#caps_eps = Uniform(-delta,delta)

# draw one epsilon per facility
#eps = rand(dist_eps, nloc)        # vector of length nloc
#eps1 = rand(caps_eps, nloc)        # vector of length nloc

# form the perturbed fixed‐costs
#fixedcost_perturbed = fixedcost .* (1 .+ eps)
#capacity_perturbed = fixedcost.*(1 .+ eps1);

# if you want integers:
# fixedcost = round.(Int, fixedcost_perturbed)
# capacity = round.(Int, capacity_perturbed);


### Standard deviation

In [2]:
theta = rand(Uniform(0, 0.5), 50)
#theta = 1
demandstdev = theta .* demandmean;

### Hyperparameters

In [3]:
unmet_pen = 50
scaling_factor = 0.000015;
bernoulli_case = true # true gives demand as bernoulli*normal, false gives normal

true

### Facility Location 

In [4]:
function facility_location(nloc,ncust,capacity,cost, demand)
    model = Model(Gurobi.Optimizer)
    # Decision Variables
    @variable(model,x[1:nloc], Bin)
    @variable(model,y[1:nloc,1:ncust]>=0)
    @variable(model,shortfall[1:ncust]>=0)

    #constraints
    @constraint(model,[j in 1:ncust], sum(y[i,j] for i in 1:nloc) + shortfall[j] ==  demand[j])
    @constraint(model,[i in 1:nloc], sum(y[i,j] for j in 1:ncust) - capacity[i]*x[i]<=0 )

    #Objective function
    @objective(model, Min, sum(fixedcost[i]*x[i] for i in 1:nloc) + scaling_factor* sum(cost[i,j]*y[i,j] for i in 1:nloc for j in 1:ncust) + sum(unmet_pen*shortfall[j] for j in 1:ncust))

    optimize!(model)

    return objective_value(model),collect(value.(x))
end

facility_location (generic function with 1 method)

### Sampling

In [5]:
function sampling(nscen,nloc,demandmean,demandstdev)
    ncust = length(demandmean)
    xsolutions = zeros(nscen,nloc)
    #pcost = zeros(nscen)
    demand = zeros(ncust)
    #ratio = zeros(nscen)
    for scenario in 1:nscen
        globaldem = round(max(rand(Normal(1000,300)),0),digits=0)
        for i in 1:ncust
            if bernoulli_case
                Bernoulli_demand = rand(Bernoulli(0.5))
            else
                Bernoulli_demand = 1
            end
            demand[i]=Bernoulli_demand*(globaldem+round(max(rand(Normal(demandmean[i],demandstdev[i])),0),digits=0))
        end
        _,xsolutions[scenario,:] = facility_location(nloc,ncust,capacity,cost,demand)
    end
    return xsolutions
end
xsolutions = sampling(30,25,demandmean,demandstdev);

Set parameter Username
Set parameter LicenseID to value 2711083
Academic license - for non-commercial use only - expires 2026-09-20
Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (mac64[arm] - Darwin 24.6.0 24G222)

CPU model: Apple M3
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 75 rows, 1325 columns and 2575 nonzeros
Model fingerprint: 0x4f8854d5
Variable types: 1300 continuous, 25 integer (25 binary)
Coefficient statistics:
  Matrix range     [1e+00, 2e+04]
  Objective range  [8e-03, 8e+03]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e+03, 8e+03]
Found heuristic solution: objective 2995650.0000
Presolve removed 22 rows and 573 columns
Presolve time: 0.00s
Presolved: 53 rows, 752 columns, 1452 nonzeros
Variable types: 728 continuous, 24 integer (24 binary)

Root relaxation: objective 5.020058e+04, 89 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 E

### (Optional) Code to check number of unique solutions 

In [6]:
function n_unique_rows(A::AbstractMatrix)
    # turn every row into an immutable tuple
    rows_as_tuples = (Tuple(view(A, i, :)) for i in axes(A,1))
    uniq = unique(rows_as_tuples)     # removes duplicates
    return uniq, length(uniq)
end
_,u1= n_unique_rows(xsolutions);
u1


30

### Set of scenarios

In [7]:
nsamples = 100   ### |S|: The set of scenarios 
ncust = 50
online_samples = zeros(nsamples, ncust)
for s in 1:nsamples
    globaldem = round(max(rand(Normal(1000,300)),0),digits=0)
    for j in 1:ncust
        if bernoulli_case
            Bernoulli_demand = rand(Bernoulli(0.5))
        else
            Bernoulli_demand = 1
        end
        online_samples[s,j] = Bernoulli_demand*(globaldem+round(max(rand(Normal(demandmean[j],demandstdev[j])),0),digits=0))
    end
end

### Decoupling of continuous and binary variable

In [8]:
# The function takes set of solutions as solved_x and set of scenarios as demand, and then return a matrix 
# that contains objective solutions for each scenario solution pair
function solve_for_continuous_var(nloc,ncust,capacity,cost,demand,solved_x)
    model = Model(Gurobi.Optimizer)
    # Decision Variables
    @variable(model,x[1:nloc], Bin)
    @variable(model,y[1:nloc,1:ncust]>=0)
    @variable(model,shortfall[1:ncust]>=0)

    #constraints
    @constraint(model,[j in 1:ncust], sum(y[i,j] for i in 1:nloc) + shortfall[j] == demand[j])
    @constraint(model,[i in 1:nloc], sum(y[i,j] for j in 1:ncust) - capacity[i]*x[i]<=0 )
    @constraint(model, x == solved_x)  # fixing the binary decision variable

    #Objective function
    @objective(model, Min, sum(fixedcost[i]*x[i] for i in 1:nloc) + scaling_factor*sum(cost[i,j]y[i,j] for i in 1:nloc for j in 1:ncust) + sum(unmet_pen*shortfall[j] for j in 1:ncust))

    optimize!(model)
    if termination_status(model) == MOI.OPTIMAL
        return objective_value(model), sum(fixedcost[i]*value.(x[i]) for i in 1:nloc)/objective_value(model)
    else
        return -1,-1
    end
end
#y = solve_for_continuous_var(8,10,capacity,cost,demandmean, xsolutions[1,:]) 
function Calculations_For_Extensive_Form(nloc,ncust,capacity,cost,online_samples, xsolutions)
    total_scenarios = size(online_samples,1)
    total_solutions = size(xsolutions,1)
    obj_value = zeros(total_scenarios,total_solutions)
    ratio = zeros(total_scenarios,total_solutions)
    for scenario in 1:total_scenarios
        demand = online_samples[scenario,:]
        for solution in 1:total_solutions
            obj_value[scenario,solution], ratio[scenario,solution] = solve_for_continuous_var(nloc,ncust,capacity,cost,demand,xsolutions[solution,:])
        end
    end
    return obj_value,ratio
end

obj_matrix,r = Calculations_For_Extensive_Form(25,50,capacity,cost,online_samples,xsolutions);


Set parameter Username
Set parameter LicenseID to value 2711083
Academic license - for non-commercial use only - expires 2026-09-20
Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (mac64[arm] - Darwin 24.6.0 24G222)

CPU model: Apple M3
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 100 rows, 1325 columns and 2600 nonzeros
Model fingerprint: 0x4de805a1
Variable types: 1300 continuous, 25 integer (25 binary)
Coefficient statistics:
  Matrix range     [1e+00, 2e+04]
  Objective range  [8e-03, 8e+03]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e+00, 9e+03]
Found heuristic solution: objective 2899650.0000
Presolve removed 74 rows and 1237 columns
Presolve time: 0.00s
Presolved: 26 rows, 88 columns, 176 nonzeros
Variable types: 88 continuous, 0 integer (0 binary)

Root relaxation: objective 4.220392e+04, 13 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl

In [9]:
n_neg1 = count(x -> x == -1, obj_matrix)

0

### Reformulated Extensive Form

In [11]:
function Reformulated_Extensive_Form(S,l,K,obj_value)
    # Initialize model
    model = Model(Gurobi.Optimizer)
    MOI.set(model, MOI.TimeLimitSec(), 30.0)

    #Total scenarios

    # Decision Variables
    @variable(model, z[1:l], Bin)
    @variable(model, w[1:l , 1:S], Bin)

    # Constraints
    for scenario in 1:S
        for solution in 1:l
            if obj_value[scenario,solution] == -1
                @constraint(model, w[solution,scenario] == 0)   ## weights of infeasible scenario/first stage decision pair is 0
            end
        end
    end


    for scenario in 1:S
        for solution in 1:l
            if obj_value[scenario,solution] == -1
                @constraint(model, w[solution,scenario] == 0)   ## weights of infeasible scenario/first stage decision pair is 0
            end
        end
    end
            
        
    for j in 1:S
        @constraint(model, [i in 1:l], w[i,j] <= z[i])
        @constraint(model, sum(w[i,j] for i in 1:l) == 1)
    end
    @constraint(model, sum(z[i] for i in 1:l) == K)

    # Objective
    objective = 0.0
    for scenario in 1:S
        for solution in 1:l
            objective += w[solution,scenario]*obj_value[scenario,solution]
        end
    end

    # Define the objective function
    @objective(model, Min, objective/S)

    # Solve the model
    #set_silent(model)
    optimize!(model)
    # Check solver status

    if termination_status(model) == MOI.OPTIMAL
        return objective_value(model),value.(z)
    else
        return -1,[]
    end    
end 
Solution,z=Reformulated_Extensive_Form(100,30,5,obj_matrix);

Set parameter Username
Set parameter LicenseID to value 2711083
Academic license - for non-commercial use only - expires 2026-09-20
Set parameter TimeLimit to value 30
Set parameter TimeLimit to value 30
Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (mac64[arm] - Darwin 24.6.0 24G222)

CPU model: Apple M3
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Non-default parameters:
TimeLimit  30

Optimize a model with 3101 rows, 3030 columns and 9030 nonzeros
Model fingerprint: 0xf0386fd5
Variable types: 0 continuous, 3030 integer (3030 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [2e+02, 2e+04]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e+00, 5e+00]
Presolve time: 0.01s
Presolved: 3101 rows, 3030 columns, 9030 nonzeros
Variable types: 0 continuous, 3030 integer (3030 binary)
Found heuristic solution: objective 55660.603033

Root relaxation: objective 5.129175e+04, 412 iterations, 0.00 seconds (0.01 work unit

### Test Sample

In [12]:
nsamples = 300
ncust = 50
test_samples = zeros(nsamples, ncust)
for s in 1:nsamples
    globaldem = round(max(rand(Normal(1000,300)),0),digits=0)
    for j in 1:ncust
        if bernoulli_case
            Bernoulli_demand = rand(Bernoulli(0.5))
        else
            Bernoulli_demand = 1
        end
        test_samples[s,j] = Bernoulli_demand*(globaldem+round(max(rand(Normal(demandmean[j],demandstdev[j])),0),digits=0))
    end
end

### Offline Calculation of Test Data

In [13]:
function OfflineTest(nloc,ncust,capacity,cost,test_samples)
    num_rows = size(test_samples, 1)
    offline_cost= zeros(num_rows)
    #p_cost = zeros(num_rows)

    #Offline calculations for the test data
    total_cost = 0.0
    for i in 1:num_rows
        obj_value,_ =facility_location(nloc,ncust,capacity,cost, test_samples[i,:])
        offline_cost[i] = obj_value
        total_cost += obj_value
        #p_cost[i] = p
    end
    total_offline_cost = total_cost/num_rows

    return total_offline_cost
end
offline_cost = OfflineTest(25,50,capacity,cost,test_samples);

Set parameter Username
Set parameter LicenseID to value 2711083
Academic license - for non-commercial use only - expires 2026-09-20
Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (mac64[arm] - Darwin 24.6.0 24G222)

CPU model: Apple M3
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 75 rows, 1325 columns and 2575 nonzeros
Model fingerprint: 0x56ce4745
Variable types: 1300 continuous, 25 integer (25 binary)
Coefficient statistics:
  Matrix range     [1e+00, 2e+04]
  Objective range  [8e-03, 8e+03]
  Bounds range     [0e+00, 0e+00]
  RHS range        [6e+02, 1e+04]
Found heuristic solution: objective 2942350.0000
Presolve removed 19 rows and 495 columns
Presolve time: 0.00s
Presolved: 56 rows, 830 columns, 1605 nonzeros
Variable types: 806 continuous, 24 integer (24 binary)

Root relaxation: objective 6.027311e+04, 121 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 

### Second Stage

In [14]:
## Function that takes the z binary vector returned by model and solves the second stage for a given realization
function Second_Stage_Cost(nloc, ncust,cost,capacity, demand,xsolutions,z)
    model = Model(Gurobi.Optimizer)
    # Decision Variables
    @variable(model,x[1:nloc], Bin)
    @variable(model,y[1:nloc,1:ncust]>=0)
    @variable(model,shortfall[1:ncust]>=0)
    @variable(model, w[1:length(z)], Bin)

    #constraints
    @constraint(model,[j in 1:ncust], sum(y[i,j] for i in 1:nloc) + shortfall[j] == demand[j])
    @constraint(model,[i in 1:nloc], sum(y[i,j] for j in 1:ncust) - capacity[i]*x[i]<=0 )

    @constraint(model, [i in 1:length(z)], w[i] <= z[i])
    @constraint(model, sum(w[i] for i in 1:length(z)) == 1)
    @constraint(model, x == sum(w[i] * xsolutions[i,:] for i in 1:length(z)))


    #Objective function
    @objective(model, Min, sum(fixedcost[i]*x[i] for i in 1:nloc) + scaling_factor*sum(cost[i,j]y[i,j] for i in 1:nloc for j in 1:ncust) + sum(unmet_pen*shortfall[j] for j in 1:ncust))

    optimize!(model)

    return objective_value(model), sum(unmet_pen*value.(shortfall[j]) for j in 1:ncust)
end

Second_Stage_Cost (generic function with 1 method)

### Evaluation

In [15]:
function Evaluation(nloc,ncust,cost,capacity,test_sample,x_solutions,z)
    num_rows = size(test_sample, 1)
    p_cost = zeros(num_rows) # Stores Penalty cost for each sample
    m_cost = zeros(num_rows)  # Stores model cost for each sample
    ### Second stage cost
    model_cost = 0.0
    for index in 1:num_rows
        model_obj_value,p = Second_Stage_Cost(nloc, ncust, cost,capacity,test_sample[index,:], x_solutions,z)
        model_cost += model_obj_value
        p_cost[index] = p
        m_cost[index] = model_obj_value
    end
    total_model_cost = model_cost / num_rows

    return total_model_cost,p_cost,m_cost
end 
modelcost,pcos,mcost = Evaluation(25,50,cost,capacity,test_samples,xsolutions,z);

Set parameter Username
Set parameter LicenseID to value 2711083
Academic license - for non-commercial use only - expires 2026-09-20
Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (mac64[arm] - Darwin 24.6.0 24G222)

CPU model: Apple M3
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 131 rows, 1355 columns and 2788 nonzeros
Model fingerprint: 0x53c4d4b0
Variable types: 1300 continuous, 55 integer (55 binary)
Coefficient statistics:
  Matrix range     [1e+00, 2e+04]
  Objective range  [8e-03, 8e+03]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e+00, 1e+04]
Found heuristic solution: objective 2957350.0000
Presolve removed 91 rows and 1133 columns
Presolve time: 0.00s
Presolved: 40 rows, 222 columns, 447 nonzeros
Variable types: 217 continuous, 5 integer (5 binary)

Root relaxation: objective 6.501283e+04, 39 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Ex

In [336]:
penalty_pos=Float64[]
penalty_val = Float64[]
for i in 1:length(pcos)
    if pcos[i] >0.1
        push!(penalty_pos,i)
       push!(penalty_val,pcos[i])
    end
end
df1 = DataFrame(
    Scenario = penalty_pos,
    Penalty = penalty_val
)
df1


Row,Scenario,Penalty
,Float64,Float64
1,134.0,2800.0
2,178.0,5200.0
3,237.0,6300.0
4,241.0,300.0


In [17]:
offline_cost

53615.44191091769

In [16]:
modelcost

55371.43850012124

### Gap between offline cost and model cost

In [18]:
(modelcost - offline_cost)/offline_cost * 100

3.275169478452031

### Select z

In [44]:
### Function that returns the z vector, i.e., K solutions from the given list of precomputed solutions
function obtain_policy(nloc,ncust,capacity,cost,online_samples,xsolutions,K)
    S = size(online_samples, 1)   # number of test scenarios
    l = size(xsolutions, 1)      # number of offline candidates

    obj_matrix,_ = Calculations_For_Extensive_Form(nloc,ncust,capacity,cost,online_samples, xsolutions)
    
    _,z = Reformulated_Extensive_Form(S,l,K,obj_matrix)
    return z
end

obtain_policy (generic function with 1 method)

### Assign Partitions

In [170]:
### Given a list of precomputed solutions, z vector and list of scenarios, the function returns the K partitions of the set of scenarios
function assign_partitions(nloc,ncust,capacity,cost,z,xsolutions, online_samples)
    # Determine how many poilicies are active, i.e., k
    selected_indices = findall(v -> v ≥ 0.5, z)
    K = length(selected_indices)
    # Total scenarios
    S = size(online_samples, 1)
    # prepare output
    partitions = [Int[] for _ in 1:K]

    # for each scenario, pick the best selected policy
    for j in 1:S
        sample = online_samples[j, :]
        best_val = Inf
        best_k   = 0
        for (k_idx, sol_idx) in enumerate(selected_indices)
            val, _ = solve_for_continuous_var(nloc,ncust,capacity,cost,sample,xsolutions[sol_idx,:])
            if val < best_val
                best_val = val
                best_k   = k_idx
            end
        end
        push!(partitions[best_k], j)
    end

    return partitions
end
partitions = assign_partitions(25,50,capacity,cost,z,xsolutions,online_samples);

### Solving the 2-stage

In [46]:
function extensive_form(nloc,ncust,capacity,cost,partition,online_samples)

    model = Model(Gurobi.Optimizer)
    nscen = length(partition)

    # Decision Variables
    @variable(model,x[1:nloc], Bin)
    @variable(model,y[1:nloc,1:ncust,1:nscen]>=0)
    @variable(model,shortfall[1:ncust,1:nscen]>=0)

    #constraints
    for k in 1:nscen
        scenario_index = partition[k]
        @constraint(model,[j in 1:ncust], sum(y[i,j,k] for i in 1:nloc) + shortfall[j] ==  online_samples[scenario_index,j])
        @constraint(model,[i in 1:nloc], sum(y[i,j,k] for j in 1:ncust) - capacity[i]*x[i]<=0 )
    end

    unmet_pen = 50

    #Objective function
    @objective(model, Min, sum(fixedcost[i]*x[i] for i in 1:nloc) + 0.000015/nscen* sum(cost[i,j]*y[i,j,k] for i in 1:nloc for j in 1:ncust for k in 1:nscen) + 1/nscen* sum(unmet_pen*shortfall[j,k] for j in 1:ncust for k in 1:nscen))

    optimize!(model)

    return collect(value.(x))
end

extensive_form (generic function with 1 method)

### L-Augmentation Heuristic

In [177]:
function Augmentation_Heuristic(nloc, ncust,capacity,cost,online_samples, xsolutions, max_iter, K)
    # Calculate unique number of solutions
    _,u1= n_unique_rows(xsolutions)
    # Initialize L
    L = copy(xsolutions)
    count =0
    for i in 1:max_iter
        count += 1
        L_prev = copy(L)
        # Obtain the K candidates through z Vector
        z = obtain_policy(nloc,ncust,capacity,cost,online_samples,L,K)
        # Generate partitions 
        partitions = assign_partitions(nloc,ncust,capacity,cost,z,L,online_samples)
        Y_new = zeros(K,nloc)
        for (i,part) in enumerate(partitions)
            Y_new[i,:] = extensive_form(nloc,ncust,capacity,cost,part,online_samples)
        end
        L = unique(vcat(L_prev, Y_new), dims=1)
        _,u2= n_unique_rows(L)
        if size(L,1) == size(L_prev,1)
            return L,z,count  # break the loop if no new solution found
        end
        u1 = u2
    end
    _,u2 = n_unique_rows(L)
    znew = obtain_policy(nloc,ncust,capacity,cost,online_samples,L,K)
    selected_indices = findall(znew > 0.9)
    return L, znew, count
end
candidates,znew, cnt = Augmentation_Heuristic(25,50,capacity,cost,online_samples,xsolutions,10, 5);


### Evaluating the heuristic

In [178]:
heuristiccost,p1,m1 = Evaluation(25,50,cost,capacity,test_samples,candidates,znew);

### Number of iterations

In [179]:
cnt

3

### New Candidates list after heuristic is run

In [180]:
candidates

38×25 Matrix{Float64}:
 -0.0  -0.0   0.0  -0.0  -0.0   0.0  …   0.0  -0.0   0.0  -0.0  -0.0  -0.0
  0.0  -0.0  -0.0  -0.0  -0.0   0.0      0.0  -0.0   0.0   0.0   0.0   0.0
 -0.0  -0.0  -0.0  -0.0  -0.0  -0.0     -0.0  -0.0  -0.0   1.0  -0.0  -0.0
 -0.0   1.0  -0.0  -0.0  -0.0  -0.0      1.0  -0.0  -0.0  -0.0  -0.0   0.0
  0.0   0.0   0.0  -0.0  -0.0  -0.0      0.0  -0.0   0.0   1.0   0.0   0.0
 -0.0   0.0  -0.0  -0.0  -0.0  -0.0  …  -0.0  -0.0  -0.0   0.0  -0.0  -0.0
 -0.0  -0.0  -0.0   1.0  -0.0   1.0     -0.0  -0.0  -0.0  -0.0  -0.0  -0.0
 -0.0  -0.0  -0.0  -0.0  -0.0  -0.0     -0.0  -0.0  -0.0  -0.0  -0.0  -0.0
 -0.0  -0.0   0.0   1.0  -0.0   0.0      0.0   0.0   0.0   1.0   0.0  -0.0
  0.0   0.0   0.0   1.0  -0.0   0.0     -0.0  -0.0   0.0   0.0   0.0   0.0
  ⋮                             ⋮    ⋱         ⋮                      
 -0.0  -0.0  -0.0  -0.0  -0.0   1.0     -0.0  -0.0  -0.0   1.0  -0.0  -0.0
  0.0  -0.0  -0.0   0.0  -0.0  -0.0  …  -0.0  -0.0  -0.0  -0.0  -0.0   0.0
 -0.0 

### L-Augmentation cost

In [181]:
heuristiccost

54175.117051344576

### Gap 

In [182]:
(heuristiccost - offline_cost)/offline_cost * 100

4.432645582804233